# 0 Imports

In [31]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd

from src.suporte import funcoes_suporte as fs

## 0.1 Funções Suporte

In [32]:
fs.jupyter_settings(altura = 10, largura = 12, fonte = 8)
fs.supressao_notacao(casa_decimal = 2)

# 0.2 Load Data

In [33]:
df_all = fs.load_pickle("../data/interim/2.all_quantity.pkl")
df_all.sample(5)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
482222,577438,22475,SKULL DESIGN TV DINNER TRAY,3,2011-11-20,4.95,18118,United Kingdom
338166,566495,21992,VINTAGE PAISLEY STATIONERY SET,2,2011-09-13,1.25,15529,United Kingdom
67780,541845,22079,RIBBON REEL HEARTS DESIGN,5,2011-01-23,1.65,15167,United Kingdom
388659,570448,21817,GLITTER CHRISTMAS TREE,36,2011-10-10,0.39,13224,United Kingdom
377468,569555,23208,LUNCH BAG VINTAGE LEAF DESIGN,10,2011-10-05,1.65,17728,United Kingdom


In [34]:
df_compras = fs.load_pickle("../data/interim/2.df_compras.pkl")
df_compras.sample(5)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
88829,543820,84459B,YELLOW METAL CHICKEN HEART,12,2011-02-13,1.49,16084,United Kingdom
383673,570082,23559,WOODLAND BUNNIES LOLLY MAKERS,30,2011-10-07,2.08,12524,Germany
450131,575166,22577,WOODEN HEART CHRISTMAS SCANDINAVIAN,24,2011-11-08,0.29,17022,United Kingdom
279229,561247,23244,ROUND STORAGE TIN VINTAGE LEAF,6,2011-07-26,1.95,16592,United Kingdom
158273,550280,22188,BLACK HEART CARD HOLDER,4,2011-04-15,3.95,13048,United Kingdom


In [35]:
df_compras.columns

Index(['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date',
       'unit_price', 'customer_id', 'country'],
      dtype='object')

In [36]:
df_returns = fs.load_pickle("../data/interim/2.df_returns.pkl")
df_returns.sample(5)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
311400,C564281,21232,STRAWBERRY CERAMIC TRINKET BOX,-1,2011-08-24,1.25,14796,United Kingdom
433655,C573949,23013,GLASS APOTHECARY BOTTLE TONIC,-4,2011-11-02,3.95,14511,United Kingdom
465413,C576229,23236,STORAGE TIN VINTAGE DOILY,-3,2011-11-14,2.89,13816,Germany
353161,C567767,22925,BLUE GIANT GARDEN THERMOMETER,-4,2011-09-22,5.95,17460,United Kingdom
468957,C576561,23146,TRIPLE HOOK ANTIQUE IVORY ROSE,-1,2011-11-15,2.89,15311,United Kingdom


# 1.0 F.E.

In [37]:
df_ref = df_compras.drop(columns = ['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'country']).drop_duplicates( ignore_index=True)
df_ref.head()

,customer_id
0,17850
1,13047
2,12583
3,13748
4,15100


## 1.1 Gross Revenue (Faturamento) quantidade * preço

In [38]:
df_compras["faturamento"] = df_compras["quantity"] * df_compras["unit_price"]

## 1.2 Monetário

In [39]:
df_monetario = df_compras[["customer_id", "faturamento"]].groupby("customer_id").sum().reset_index()

df_ref = pd.merge(df_ref, df_monetario, how="left", on="customer_id")

df_ref.sample()

,customer_id,faturamento
3345,16400,303.93


In [40]:
df_ref.isna().sum()

customer_id    0
faturamento    0
dtype: int64

In [41]:
del df_monetario

## 1.3 Recência

In [42]:
df_recencia = df_compras[["customer_id", "invoice_date"]].groupby("customer_id").max().reset_index()

df_recencia["recencia_days"] = (df_compras["invoice_date"].max() - df_recencia["invoice_date"]).dt.days

df_recencia = df_recencia[["customer_id", "recencia_days"]].copy()

df_ref = pd.merge(df_ref, df_recencia, how = "left", on="customer_id")

df_ref.head()

,customer_id,faturamento,recencia_days
0,17850,5391.21,372
1,13047,3232.59,56
2,12583,6705.38,2
3,13748,948.25,95
4,15100,876.00,333


In [43]:
df_ref.isna().sum()

customer_id      0
faturamento      0
recencia_days    0
dtype: int64

In [44]:
del df_recencia

## 1.4 Frequência

In [45]:
df_frequencia = df_compras[["customer_id", "invoice_no"]].drop_duplicates().groupby("customer_id").count().reset_index().rename(columns = {"invoice_no": "frequencia"})

df_ref = pd.merge(df_ref, df_frequencia, how='left', on='customer_id')

df_ref.head()

,customer_id,faturamento,recencia_days,frequencia
0,17850,5391.21,372,34
1,13047,3232.59,56,9
2,12583,6705.38,2,15
3,13748,948.25,95,5
4,15100,876.00,333,3


In [46]:
df_ref.isna().sum()

customer_id      0
faturamento      0
recencia_days    0
frequencia       0
dtype: int64

In [47]:
del df_frequencia

## 1.4 Avg Ticket

In [48]:
avg_ticket = df_compras[['customer_id','faturamento']].groupby('customer_id').mean().reset_index().rename(columns = {'faturamento':'avg_faturamento'})
df_ref = pd.merge(df_ref, avg_ticket, how='left', on='customer_id')
df_ref.head()

,customer_id,faturamento,recencia_days,frequencia,avg_faturamento
0,17850,5391.21,372,34,18.15
1,13047,3232.59,56,9,18.90
2,12583,6705.38,2,15,28.90
3,13748,948.25,95,5,33.87
4,15100,876.00,333,3,292.00


In [49]:
df_ref.isna().sum()

customer_id        0
faturamento        0
recencia_days      0
frequencia         0
avg_faturamento    0
dtype: int64

## 1.5. Returns

In [50]:
df_ret = df_returns[['customer_id', 'invoice_no']].drop_duplicates().groupby('customer_id').count().reset_index().rename(columns ={'invoice_no': 'retornos'})
df_ref = pd.merge(df_ref, df_ret, how='left', on='customer_id')
df_ref.loc[df_ref['retornos'].isna(),'retornos'] = 0
df_ref.head()

,customer_id,faturamento,recencia_days,frequencia,avg_faturamento,retornos
0,17850,5391.21,372,34,18.15,1.00
1,13047,3232.59,56,9,18.90,7.00
2,12583,6705.38,2,15,28.90,2.00
3,13748,948.25,95,5,33.87,0.00
4,15100,876.00,333,3,292.00,3.00


In [51]:
df_ref.isna().sum()

customer_id        0
faturamento        0
recencia_days      0
frequencia         0
avg_faturamento    0
retornos           0
dtype: int64

# 2.0 Exportar DF

In [52]:
path = "../data/interim/3.fe.pkl"
fs.save_pickle(obj=df_ref,path=path)